In [ ]:
# All Libraries needed - Time - 10 secs
!pip install transformers>=4.41 accelerate>=0.30 bitsandbytes>=0.43 sentencepiece>=0.2 datasets==2.20.0 tqdm==4.66.4 pydantic>=2.6

In [ ]:
# ===== Standard Library ===== Time - 14 secs
import os
import json
import re
import concurrent.futures
from typing import List, Tuple

# ===== PyTorch =====
import torch

# ===== Progress & Utilities =====
from tqdm.auto import tqdm

# ===== Hugging Face Ecosystem =====
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from transformers.utils import logging

# ===== Environment Configuration =====
# Enable memory-efficient CUDA allocations (allows gradual growth)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Hugging Face Hub authentication token
# NOTE: Move this to shell env or .env file before pushing to GitHub
os.environ["HF_TOKEN"] = ""

# ===== CUDA Optimization (T4 GPU) =====
# Flash attention not supported on T4; use memory-efficient variant
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
# Math fallback prevents "Invalid backend" error on unsupported ops
torch.backends.cuda.enable_math_sdp(True)

# ===== Suppress Verbose Logging =====
# Reduces noise from transformers library during model loading
logging.set_verbosity_error()

In [ ]:
# ===== Model & Tokenizer Loading ===== Time - 1 min
# Load Qwen2.5-1.5B-Instruct: lightweight instruction-tuned model suitable for T4 GPU

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Set pad token to avoid warnings during generation
tokenizer.pad_token = tokenizer.eos_token

# Load model with 4-bit quantization and float16 for memory efficiency
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",              # Automatically distribute across available devices
    load_in_4bit=True,              # 4-bit quantization to fit on T4 (~6GB VRAM)
    torch_dtype=torch.float16       # Use float16 for faster inference
)

# Set model to evaluation mode (disables dropout, activates batch norm eval)
model.eval()

print(f"✓ Model {MODEL_ID} loaded successfully")
print(f"✓ Model device: {next(model.parameters()).device}")

In [ ]:
# ===== Text Processing Functions =====

def load_articles_from_json(file_path: str) -> Dataset:
    """
    Load articles from a JSON file with artifact_data field.
    
    Args:
        file_path: Path to JSON file
    
    Returns:
        HuggingFace Dataset with id, content, platform, author_id, author_full_name, link
    """
    with open(file_path, "r") as file:
        data = json.load(file)

    return Dataset.from_dict({
        "id": [item["id"] for item in data["artifact_data"]],
        "content": [item["content"] for item in data["artifact_data"]],
        "platform": [item["platform"] for item in data["artifact_data"]],
        "author_id": [item["author_id"] for item in data["artifact_data"]],
        "author_full_name": [item["author_full_name"] for item in data["artifact_data"]],
        "link": [item["link"] for item in data["artifact_data"]],
    })


def clean_text(text: str) -> str:
    """
    Clean text by removing special characters and normalizing whitespace.
    
    Args:
        text: Raw text string
    
    Returns:
        Cleaned text with normalized spacing
    """
    # Remove special characters; keep alphanumeric, punctuation, and apostrophes
    text = re.sub(r"[^\w\s.,!?']", " ", text)
    # Collapse multiple spaces to single space
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
# ===== JSON Parsing & Tokenization Utilities =====

def extract_first_json_object(text: str) -> dict:
    """
    Extract and parse the FIRST valid JSON object from text.
    Ignores any trailing or leading junk.
    
    Args:
        text: Text containing JSON object
    
    Returns:
        Parsed JSON object as dict
    
    Raises:
        ValueError: If no valid JSON object found
    """
    decoder = json.JSONDecoder()
    text = text.strip()

    # Scan for first '{' and try to parse from there
    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, end = decoder.raw_decode(text[i:])
                return obj
            except json.JSONDecodeError:
                continue

    raise ValueError("No valid JSON object found")


def truncate_to_max_tokens(text: str, tokenizer, max_tokens: int = 512) -> str:
    """
    Truncate text to maximum token count using tokenizer.
    Safety mechanism to prevent OOM on T4 GPU.
    
    Args:
        text: Input text
        tokenizer: HuggingFace tokenizer
        max_tokens: Maximum number of tokens (default: 512)
    
    Returns:
        Truncated and decoded text
    """
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=max_tokens,
        return_tensors="pt"
    )
    return tokenizer.decode(tokens["input_ids"][0], skip_special_tokens=True)

In [ ]:
# ===== Core Pipeline: Instruction-Answer Pair Generation =====

def generate_instruction_answer_pairs(
    extract: str,
    tokenizer,
    model,
    max_new_tokens: int = 256,
    temperature: float = 0.7
) -> List[Tuple[str, str]]:
    """
    Generate instruction-answer pairs from a text extract using LLM.
    
    Args:
        extract: Text extract to generate pairs from
        tokenizer: HuggingFace tokenizer
        model: HuggingFace language model
        max_new_tokens: Max tokens for generation (default: 128)
        temperature: Sampling temperature (default: 0.7)
    
    Returns:
        List of (instruction, answer) tuples
    """
    # HARD SAFETY CAP: Truncate extract to prevent OOM on T4
    extract = truncate_to_max_tokens(
        extract,
        tokenizer,
        max_tokens=512  # Non-negotiable limit for T4
    )
    
    # Construct detailed prompt with rules and examples
    prompt = f"""Based on the following extract, generate three instruction-answer pairs.

Rules:
- Each instruction must ask to write about a specific topic contained in the extract.
- Instructions must be self-contained and general.
- Answers must be factual and grounded ONLY in the extract.
- Answers must imitate the writing style of the extract.
- Do NOT mention the extract or context explicitly.

Output strictly in JSON with the following structure:

{{
  "instruction_answer_pairs": [
    {{"instruction": "...", "answer": "..."}},
    {{"instruction": "...", "answer": "..."}},
    {{"instruction": "...", "answer": "..."}}
  ]
}}

Extract:
{extract}
"""

    # Format as chat messages
    messages = [
        {"role": "system", "content": "You are an expert at creating high-quality instruction-answer pairs from text."},
        {"role": "user", "content": prompt},
    ]

    # Tokenize and move to model device
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt", 
        add_generation_prompt=True
    ).to(model.device)

    # Generate response
    outputs = model.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # CRITICAL FIX: Extract only the NEW tokens generated by the model
    # Skip the input tokens to avoid getting the prompt back
    new_tokens = outputs[0][inputs.shape[1]:]
    # Decode output
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Parse JSON response
    try:
        data = extract_first_json_object(decoded)
        pairs = [
            (p["instruction"], p["answer"])
            for p in data.get("instruction_answer_pairs", [])
        ]
        return pairs
    except Exception as e:
        print(f"⚠️  JSON parse failed for extract: {str(e)[:50]}...")
        return []

In [ ]:
# ===== Text Extraction & Chunking =====

def extract_substrings(
    dataset: Dataset,
    text_column: str = "content",
    min_length: int = 1000,
    max_length: int = 2000
) -> List[str]:
    """
    Extract sentence-based chunks from dataset texts within min/max length bounds.
    
    Args:
        dataset: HuggingFace Dataset
        text_column: Name of text column (default: "content")
        min_length: Minimum chunk length in characters (default: 1000)
        max_length: Maximum chunk length in characters (default: 2000)
    
    Returns:
        List of text chunks (extracts)
    """
    extracts = []
    # Regex pattern to split on sentence boundaries (., !, ?)
    sentence_pattern = r"(?<!\w\.\w.)(?<![A-Z][a-z]\.) (?<=\.|\?|!)\s"

    for article in dataset[text_column]:
        cleaned_article = clean_text(article)
        sentences = re.split(sentence_pattern, cleaned_article)
        current_chunk = ""

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            # Add sentence if it fits within max_length
            if len(current_chunk) + len(sentence) <= max_length:
                current_chunk += sentence + " "
            else:
                # Save chunk if it meets min_length
                if len(current_chunk) >= min_length:
                    extracts.append(current_chunk.strip())
                current_chunk = sentence + " "

        # Don't forget final chunk
        if len(current_chunk) >= min_length:
            extracts.append(current_chunk.strip())

    return extracts

In [ ]:
# ===== Main Pipeline: Create Instruction Dataset =====

def create_instruction_dataset(
    dataset: Dataset,
    tokenizer,
    model,
    text_column: str = "text",
    min_length: int = 200,
    max_length: int = 500
) -> Dataset:
    """
    Convert raw dataset into instruction-answer pairs.
    
    Args:
        dataset: Raw HuggingFace Dataset
        tokenizer: HuggingFace tokenizer
        model: HuggingFace language model
        text_column: Text column name (default: "text")
        min_length: Min extract length (default: 200)
        max_length: Max extract length (default: 500)
    
    Returns:
        HuggingFace Dataset with 'instruction' and 'output' columns
    """
    # Step 1: Extract text chunks
    print(f"📄 Extracting text chunks from '{text_column}' column...")
    extracts = extract_substrings(
        dataset,
        text_column=text_column,
        min_length=min_length,
        max_length=max_length
    )
    print(f"   ✓ Extracted {len(extracts)} chunks")
    
    # Step 2: Generate instruction-answer pairs for each extract
    print("🤖 Generating instruction-answer pairs...")
    instruction_answer_pairs = []
    
    for extract in tqdm(extracts, desc="Pairs"):
        pairs = generate_instruction_answer_pairs(extract, tokenizer, model)
        instruction_answer_pairs.extend(pairs)
        # Clear GPU cache after each batch to prevent OOM
        torch.cuda.empty_cache()
    
    print(f"   ✓ Generated {len(instruction_answer_pairs)} instruction-answer pairs")
    
    # Step 3: Create dataset from pairs
    if not instruction_answer_pairs:
        print("⚠️  No pairs generated; returning empty dataset")
        return Dataset.from_dict({"instruction": [], "output": []})
    
    instructions, answers = zip(*instruction_answer_pairs)
    
    return Dataset.from_dict({
        "instruction": list(instructions),
        "output": list(answers)
    })

In [ ]:
# ===== Load Raw Dataset ===== (Press Y and enter to execute)Time - 8 mins
# Using Wikipedia snapshot from 2022-03-01 (small subset for testing)

print("📥 Loading Wikipedia dataset...")
wiki = load_dataset(
    "wikipedia",
    "20220301.en",
    split="train[:10]"  # Load first 10 examples for testing
)
print(f"✓ Loaded {len(wiki)} articles")

In [ ]:
# ===== Execute Full Pipeline =====
# Generate instruction dataset from raw Wikipedia articles

print("🚀 Starting instruction dataset generation pipeline...\n")

# Create instruction dataset
instruction_dataset = create_instruction_dataset(
    wiki,
    tokenizer,
    model,
    text_column="text",
    min_length=200,
    max_length=500
)

print(f"\n✓ Instruction dataset created with {len(instruction_dataset)} examples")

# Split into train/test (90/10)
print("\n📊 Splitting dataset (90% train / 10% test)...")
split_dataset = instruction_dataset.train_test_split(test_size=0.1)
print(f"   ✓ Train: {len(split_dataset['train'])} | Test: {len(split_dataset['test'])}")

print(f"\n✓ Pipeline complete!")

In [ ]:
# ===== Display Sample Instruction-Answer Pairs =====

print("\n" + "="*80)
print("SAMPLE INSTRUCTION-ANSWER PAIRS")
print("="*80)

if len(split_dataset["train"]) > 0:
    for i in range(min(2, len(split_dataset["train"]))):
        print(f"\n--- Example {i+1} ---")
        print(f"INSTRUCTION:\n{split_dataset['train'][i]['instruction']}\n")
        print(f"OUTPUT:\n{split_dataset['train'][i]['output']}")
else:
    print("⚠️  No training examples available")

print("\n" + "="*80)
print(f"DATASET SUMMARY")
print("="*80)
print(split_dataset)